<a href="https://colab.research.google.com/github/Emil-888/Data-Science/blob/main/Copia_de_06_clasificacion_pinguinos_para_completar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Versión para completar.** Toda la teoría está. El código está vacío.
> Solución (ejercicios resueltos): `06_clasificacion_pinguinos.ipynb`.
> Teoría de clase: `06_teoria_clasificacion.pdf`. CSV: `Penguins.csv` en esta carpeta.

# UDECATALUNA — notebook para completar
_________________________________

Tema: Modelos de Aprendizaje — Clasificación con pingüinos
_________________________________

**Dataset:** `Penguins.csv` (Palmer Penguins)  
**Modelo:** `LogisticRegression` con **3 clases**

## Objetivo

Al terminar vas a poder:

* Decir qué es clasificar cuando hay **más de dos** respuestas posibles.
* Cargar, mirar y limpiar una tabla chica (faltantes, nombres, target).
* Separar train/test con `stratify` y armar un `Pipeline` **sin funciones extra**.
* Entrenar una logística y un baseline.
* Leer **tres** cosas: accuracy, matriz de confusión y reporte por clase.
* Predecir la especie de un pingüino nuevo.
* Resolver **10 ejercicios** (todos están resueltos acá; la otra notebook los deja vacíos).

> En Sydney y en churn el target era **sí / no**.  
> Acá $y$ tiene tres valores: **Adelie**, **Chinstrap**, **Gentoo**.

**Reglas (las de siempre)**

1. No entrenamos con el dataset entero.
2. Escalar y one-hot van **dentro** del `Pipeline`.
3. Primero un **baseline**, después el modelo.
4. El **test** se mira al final.
5. Con 3 clases el accuracy sirve como número global, pero el error se lee **por especie**.

Caso: un biólogo mide pico, aleta, peso, isla y sexo. Quiere saber de qué especie es el pingüino.

In [ ]:
# PASO 1 — Entorno
# Importá warnings, matplotlib.pyplot, numpy, pandas, seaborn, display.
# Silenciá warnings, tema whitegrid, imprimí la versión de pandas.


### Qué es clasificar (en una pantalla)

**Clasificar** = asignar cada fila a una **categoría**.

| Problema | Clases | Ejemplo del curso |
|---|---|---|
| Binario | 2 | casa cara / accesible · se va / se queda |
| Multiclase | 3 o más | **esta notebook**: Adelie / Chinstrap / Gentoo |

La logística sigue estimando **probabilidades**. Con 3 especies entrega tres números que suman 1, por ejemplo:

`Adelie 0.10 · Chinstrap 0.15 · Gentoo 0.75`

`predict` elige la de mayor probabilidad. No hay un umbral 0.50 como en el sí/no.

No hace falta memorizar 15 métricas. Con **tres lecturas** alcanza:

1. **Accuracy** — de cada 100, ¿cuántos acertó?
2. **Matriz de confusión** — ¿con cuáles especies se confunde?
3. **Reporte por clase** — precision y recall de *cada* especie (Chinstrap es la más chica: si el modelo la ignora, el accuracy igual puede verse “lindo”).

### Bloque 1 — Cargar y mirar

| Columna original | Nombre que usamos | Qué es |
|---|---|---|
| `species` | `especie` | **target**: Adelie, Chinstrap, Gentoo |
| `island` | `isla` | isla donde se midió |
| `bill_length_mm` | `pico_largo` | largo del pico (mm) |
| `bill_depth_mm` | `pico_ancho` | alto del pico (mm) |
| `flipper_length_mm` | `aleta` | largo de la aleta (mm) |
| `body_mass_g` | `peso` | masa (gramos) |
| `sex` | `sexo` | hembra / macho |

In [ ]:
# PASO 2 — Cargar
# raw = pd.read_csv("Penguins.csv")
# Mostrá shape, head, faltantes y value_counts de species.


### Bloque 2 — Limpieza (sin función)

Hay 2 filas casi vacías y 11 sin sexo. En un dataset de 344 filas, **tirar las incompletas** es la limpieza más honesta para empezar. No hace falta una función `limpiar_...`.

In [ ]:
# PASO 3 — Limpiar (sin función)
# dropna + rename a especie, isla, pico_largo, pico_ancho, aleta, peso, sexo.
# Imprimí cuántas filas se fueron.


### Bloque 3 — Explorar (todavía sin modelo)

Tres preguntas, en este orden:

1. **¿Qué predigo?** La especie.
2. **¿Está balanceado?** Adelie es la más frecuente, Chinstrap la más rara. No es 50/50/50 como Iris.
3. **¿Se ven grupos?** Pico y aleta suelen separar bastante a Gentoo.

In [ ]:
# PASO 4 — Dos gráficos
# countplot de especie | scatter pico_largo vs aleta, hue=especie.
# Tabla de medias por especie.


> **Para discutir.**  
> 1. Si Gentoo tiene aleta más larga, ¿el modelo puede “adivinarla” casi solo con `aleta`?  
> 2. ¿Dónde esperás más confusión: Adelie ↔ Chinstrap o Adelie ↔ Gentoo?  
> 3. `isla` no es una medida del animal. ¿La dejarías? (a veces la isla y la especie van juntas: es señal, pero no es biología del pico).

### Bloque 4 — Features y split

**Entran:** pico, aleta, peso, isla, sexo.  
**No entra:** `especie` (es la respuesta).

`stratify=y` hace que train y test copien la misma mezcla de especies. Sin eso, Chinstrap (la más rara) puede quedar casi toda de un solo lado.

In [ ]:
# PASO 5 — Split
# NUM = pico_largo, pico_ancho, aleta, peso
# CAT = isla, sexo
# train_test_split test_size=0.25, random_state=42, stratify=y
# Mostrá % de cada especie en train y en test.


### Bloque 5 — Baseline y logística

El código va **seguido**. No envolvemos nada en `armar_pipeline()` ni en `calcular_metricas()`.

* **Baseline:** `DummyClassifier(strategy="most_frequent")` → siempre dice Adelie (la más común). El Dummy **no necesita** escalar.
* **Modelo:** one-hot de isla y sexo + `StandardScaler` de las medidas + `LogisticRegression`.  
  `max_iter=1000` es para que el solver termine. El resto, default.

Con 3 clases sklearn ya usa el modo multiclase. No hay que poner nada especial.

In [ ]:
# PASO 6 — Entrenar
# DummyClassifier most_frequent → fit
# ColumnTransformer + Pipeline + LogisticRegression(max_iter=1000) → fit
# Sin funciones auxiliares.


### Bloque 6 — Las 3 métricas (y nada más)

#### 1. Accuracy

$$
\text{accuracy} = \frac{\text{aciertos}}{\text{total}}
$$

Acá las clases no están tan desbalanceadas como el churn. El accuracy **sí** se puede mirar. El baseline (siempre Adelie) va a estar cerca de 0.44: ese es el piso. Si la logística no lo pasa por mucho, no aprendió.

#### 2. Matriz de confusión

Filas = **realidad**. Columnas = **predicción**.  
La diagonal son aciertos. Fuera de la diagonal: “era A y dije B”.

#### 3. Reporte por clase

De `classification_report` nos importan, **por especie**:

* **precision** — de las veces que dije “Gentoo”, ¿cuántas lo eran?
* **recall** — de todos los Gentoo reales, ¿cuántos encontré?
* **f1** — un número que junta las dos (si una está en el piso, el f1 también).

No agregamos ROC, PR, MCC, Brier, log-loss, F2, kappa. Para un primer modelo multiclase **sobran**.

In [ ]:
# PASO 7 — Tres métricas
# accuracy del baseline y de la logística
# dos matrices de confusión
# classification_report de la logística


**Cómo se lee, en voz alta**

* Si el accuracy de la logística está cerca de 1.00 y el del baseline cerca de 0.44, el modelo está ordenando de verdad.
* Si en la matriz hay un 3 fuera de la diagonal en (Adelie, Chinstrap), decís: “tres Adelie los llamé Chinstrap”.
* Si Chinstrap tiene recall bajo, el modelo se los está comiendo (los mezcla con otra especie). Eso el accuracy global lo puede esconder.

### Bloque 7 — Tres pingüinos de ejemplo

Armamos filas **con las mismas columnas** que el train y pedimos clase + probabilidad.

In [ ]:
# PASO 8 — Tres ejemplos
# DataFrame con 3 pingüinos (mismas columnas que X).
# predict + predict_proba.


### Ejercicios (para completar)

Diez prácticas cortas. Completá cada celda. Si te trabás, mirá la solución en `06_clasificacion_pinguinos.ipynb`.

#### Ejercicio 1 — Un pingüino mezclado

Medidas de Adelie y **aleta de Gentoo** (230 mm).  
¿El modelo se va a Gentoo o se queda en Adelie? Mirá las tres probabilidades.

In [ ]:
# EJERCICIO 1 — pingüino mezclado
# Adelie en pico/peso/isla, aleta=230.
# Imprimí clase y las 3 probs.


#### Ejercicio 2 — ¿Hace falta la isla?

Mismo modelo **sin** `isla`. Si el accuracy casi no baja, la señal está en el pico y la aleta.

In [ ]:
# EJERCICIO 2 — sin isla
# Reentrená el mismo Pipeline sin la columna isla.
# Compará accuracy con el modelo full.


#### Ejercicio 3 — Baseline estratificado

`most_frequent` siempre dice Adelie.  
`stratified` tira al azar **respetando las proporciones** del train.  
Compará los dos accuracy con 1/3 (azar puro de 3 clases).

In [ ]:
# EJERCICIO 3 — dummy estratificado
# DummyClassifier(strategy="stratified", random_state=42)
# Accuracy vs most_frequent y vs 1/3.


#### Ejercicio 4 — Recall de Chinstrap a mano

1. ¿Cuántos Chinstrap reales hay en el test?  
2. ¿Cuántos de esos acertó el modelo?  
3. recall = aciertos / reales. ¿Coincide con el `classification_report`?

In [ ]:
# EJERCICIO 4 — recall a mano
# pd.crosstab(y_test, pred)
# reales y aciertos de Chinstrap → recall = aciertos / reales.


#### Ejercicio 5 — Un árbol de profundidad 3

Mismo `Pipeline`, cambiamos solo el clasificador.  
¿Gana accuracy a la logística? Un árbol se puede dibujar; la logística no.

In [ ]:
# EJERCICIO 5 — árbol
# DecisionTreeClassifier(max_depth=3, random_state=42) en el mismo estilo de Pipeline.
# Accuracy vs logística y plot_tree.


#### Ejercicio 6 — ¿Alcanza solo la aleta?

Entrenamos con **una** columna numérica.  
Si Gentoo se separa por aleta larga, el accuracy no debería irse al piso. Adelie vs Chinstrap sí se va a mezclar.

In [ ]:
# EJERCICIO 6 — solo aleta
# Entrená una logística usando únicamente la columna aleta.
# Compará accuracy y el reporte (¿Gentoo se salva? ¿Adelie/Chinstrap se pisan?).


#### Ejercicio 7 — La misma medida, otra isla

Un pingüino **idéntico** en pico, aleta, peso y sexo. Solo cambia la isla.  
¿Se mueve la probabilidad? Si sí, el modelo está usando geografía, no solo biología.

In [ ]:
# EJERCICIO 7 — misma medida, otra isla
# Tres filas idénticas salvo isla = Torgersen / Dream / Biscoe.
# Tabla de probs y predicción.


#### Ejercicio 8 — Las probabilidades tienen que sumar 1

Agarrá **una fila del test**, pedí `predict_proba` y sumá. Tiene que dar 1.000.  
Eso es softmax: tres chances que se parten el 100%.

In [ ]:
# EJERCICIO 8 — las probs suman 1
# Una fila del test: predict_proba, imprimí las 3 y la suma.


#### Ejercicio 9 — ¿A quién le erró?

Listamos las filas del test donde `pred != y_test`.  
Si la lista está vacía, en *este* corte el modelo no se equivocó. Eso puede pasar en pingüinos. No es la norma en un problema sucio (churn).

In [ ]:
# EJERCICIO 9 — a quién le erró
# Armá un DataFrame del test con columnas real y predicho.
# Filtrá donde no coinciden. ¿Cuántos son?


#### Ejercicio 10 — Split con y sin `stratify`

Mismo `random_state`, un split **sin** `stratify`.  
Compará cuántos Chinstrap cayeron en cada test. Si la diferencia es grande, ya entendiste para qué existe `stratify`.

In [ ]:
# EJERCICIO 10 — stratify sí / no
# Dos train_test_split, mismo random_state, uno con stratify=y y otro sin.
# value_counts de las especies en cada test. Chinstrap: ¿cuántos de cada lado?


### Cierre — para decir sin notebook

* Clasificar con 3 clases no es “tres sí/no sueltos”: el modelo reparte una probabilidad por especie y se queda con la más alta.
* Protocolo: dropna documentado, `stratify`, pipeline, baseline, test al final.
* Tres métricas: **accuracy** (número global), **matriz** (dónde se confunde), **reporte** (cada especie).
* Chinstrap es la clase chica: si el recall de esa fila está mal, el modelo no sirve aunque el accuracy se vea alto.
* Isla puede ayudar y a la vez “soplar” la respuesta. Una sola feature (aleta) encuentra Gentoo y mezcla las otras dos.
* Un 0.99 acá no se transfiere a un churn. El **protocolo** sí.

**UDECATALUNA** — Modelos de Aprendizaje · Clasificación · Palmer Penguins